# Módulo 5 · Notebook 3 — Patrón Multiagente

Un sistema multiagente divide el trabajo en roles especializados.

En este ejemplo usamos 3 componentes:
- **Router**: decide a qué especialista enviar la pregunta.
- **Agente Historiador**: contexto narrativo.
- **Agente Analista**: cálculos y datos concretos.

## 1. Setup

In [ ]:
import warnings
from statistics import mean

from langchain_ollama import ChatOllama

warnings.filterwarnings("ignore")

WORLD_CUP_CHAMPIONS = {
    1930: "Uruguay", 1934: "Italia", 1938: "Italia", 1950: "Uruguay",
    1954: "Alemania Federal", 1958: "Brasil", 1962: "Brasil", 1966: "Inglaterra",
    1970: "Brasil", 1974: "Alemania Federal", 1978: "Argentina", 1982: "Italia",
    1986: "Argentina", 1990: "Alemania", 1994: "Brasil", 1998: "Francia",
    2002: "Brasil", 2006: "Italia", 2010: "España",
}
GOALS_PER_WORLD_CUP = {
    1930: 70, 1934: 70, 1938: 84, 1950: 88, 1954: 140, 1958: 126,
    1962: 89, 1966: 89, 1970: 95, 1974: 97, 1978: 102, 1982: 146,
    1986: 132, 1990: 115, 1994: 141, 1998: 171, 2002: 161, 2006: 147, 2010: 145,
}

llm = ChatOllama(model="llama3.2:3b", temperature=0)
print("✅ Setup listo")

## 2. Especialistas

Implementamos dos agentes especializados como funciones.

> Nota: en sistemas productivos, cada especialista puede ser un agente completo con sus propias tools y memoria.

In [ ]:
def agente_historiador(question: str) -> str:
    prompt = f"""
Eres un historiador del fútbol.
Responde de forma narrativa y breve, en español.
Si mencionas un campeón, valida con este diccionario: {WORLD_CUP_CHAMPIONS}

Pregunta: {question}
"""
    return llm.invoke(prompt).content


def agente_analista(question: str) -> str:
    # Mini capa determinística para preguntas numéricas frecuentes
    q = question.lower()

    if "promedio" in q and "goles" in q:
        # Ejemplo fijo para mantener el notebook simple
        years = [y for y in GOALS_PER_WORLD_CUP if 1990 <= y <= 2010]
        avg = mean(GOALS_PER_WORLD_CUP[y] for y in years)
        return f"Promedio de goles entre 1990 y 2010: {avg:.2f}"

    if "campe" in q:
        # Intento simple de extraer un año
        tokens = [t for t in q.replace("?", " ").split() if t.isdigit()]
        if tokens:
            y = int(tokens[0])
            return f"Campeón {y}: {WORLD_CUP_CHAMPIONS.get(y, 'dato no disponible')}"

    prompt = f"""
Eres un analista de datos de mundiales.
Responde con enfoque cuantitativo y en español.
Usa solo estos datos:
- Campeones: {WORLD_CUP_CHAMPIONS}
- Goles por mundial: {GOALS_PER_WORLD_CUP}

Pregunta: {question}
"""
    return llm.invoke(prompt).content

print("✅ Especialistas listos")

## 3. Router y orquestador

El router decide el flujo:
- `historiador` para preguntas narrativas.
- `analista` para preguntas de cifras.
- `mixto` cuando hay ambas cosas.

In [ ]:
def route_question(question: str) -> str:
    q = question.lower()
    has_numeric_intent = any(k in q for k in ["promedio", "cuánt", "cuanto", "goles", "estad", "dato"])
    has_story_intent = any(k in q for k in ["historia", "contexto", "explica", "importancia", "por qué", "por que"])

    if has_numeric_intent and has_story_intent:
        return "mixto"
    if has_numeric_intent:
        return "analista"
    return "historiador"


def multiagent_answer(question: str) -> str:
    route = route_question(question)

    if route == "historiador":
        return f"[Ruta: historiador]\n{agente_historiador(question)}"

    if route == "analista":
        return f"[Ruta: analista]\n{agente_analista(question)}"

    # Ruta mixta: primero análisis y luego narrativa apoyada en ese resultado
    analytical = agente_analista(question)
    synthesis_prompt = f"""
Combina estas dos piezas en una sola respuesta clara y breve:
1) Resultado cuantitativo: {analytical}
2) Pregunta original: {question}

Escribe en español para nivel de curso introductorio.
"""
    narrative = llm.invoke(synthesis_prompt).content
    return f"[Ruta: mixta]\n{narrative}"

## 4. Pruebas

In [ ]:
tests = [
    "¿Quién ganó el mundial de 1978?",
    "¿Cuál fue el promedio de goles entre 1990 y 2010?",
    "Explica el contexto del mundial de 1998 y añade un dato numérico relevante.",
]

for q in tests:
    print("=" * 100)
    print("Pregunta:", q)
    print(multiagent_answer(q))
    print()

## 5. Cierre

Este patrón muestra la idea central de multiagente:
- separar responsabilidades,
- enrutar tareas,
- combinar resultados.

En una versión más avanzada puedes reemplazar el router por un LLM con salida estructurada y dar memoria individual a cada especialista.